In [5]:
import requests
import numpy as np
from lerc import decode_ma

BASE_URL = (
    "https://tiledimageservices.arcgis.com/P3ePLMYs2RVChkJx/"
    "arcgis/rest/services/"
    "USA_Transportation_Noise___Rail_Road_and_Aviation_2020/"
    "ImageServer"
)

tile_url = f"{BASE_URL}/tile/11/877/1519"

response = requests.get(tile_url)
print("Status:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Downloaded bytes:", len(response.content))

result, raster, depth, nodata = decode_ma(response.content)

print("Decode result:", result)
print("Raster dimensions:", raster.shape)
print("Data type:", raster.dtype)
print("Minimum:", raster.min())
print("Maximum:", raster.max())

Status: 200
Content type: application/octet-stream
Downloaded bytes: 156579
Decode result: 0
Raster dimensions: (256, 256)
Data type: float32
Minimum: 45.0
Maximum: 102.71997


In [6]:
from rasterio.transform import from_origin

resolution = 29.9999961896375
tile_size = 256
tile_span = resolution * tile_size

row = 877
col = 1519

left = -20037507.0672 + col * tile_span
top = 11598313.0836 - row * tile_span


West: -8371588.548912805
North: 4862953.939072105
East: -8363908.549888257
South: 4855273.9400475575


In [7]:
import requests
import numpy as np
import rasterio
from rasterio.transform import from_origin
from rasterio.mask import mask
from pyproj import Transformer
from shapely.geometry import box, mapping
from lerc import decode_ma
import math

BASE_URL = (
    "https://tiledimageservices.arcgis.com/P3ePLMYs2RVChkJx/"
    "arcgis/rest/services/"
    "USA_Transportation_Noise___Rail_Road_and_Aviation_2020/"
    "ImageServer"
)

OUTPUT = "transportation_noise.tif"


bbox_wgs84 = (-122.85, 45.43, -122.45, 45.68)
LEVEL = 11
TILE_SIZE = 256

ORIGIN_X = -20037507.0672
ORIGIN_Y = 11598313.0836

RESOLUTION = 29.9999961896375
TILE_SPAN = TILE_SIZE * RESOLUTION

transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:3857",
    always_xy=True
)

xmin, ymin = transformer.transform(
    bbox_wgs84[0], bbox_wgs84[1]
)

xmax, ymax = transformer.transform(
    bbox_wgs84[2], bbox_wgs84[3]
)

col_min = math.floor((xmin - ORIGIN_X) / TILE_SPAN)
col_max = math.floor((xmax - ORIGIN_X) / TILE_SPAN)

row_min = math.floor((ORIGIN_Y - ymax) / TILE_SPAN)
row_max = math.floor((ORIGIN_Y - ymin) / TILE_SPAN)

print("Columns:", col_min, "to", col_max)
print("Rows:", row_min, "to", row_max)

width = (col_max - col_min + 1) * TILE_SIZE
height = (row_max - row_min + 1) * TILE_SIZE

mosaic = np.full(
    (height, width),
    np.nan,
    dtype=np.float32
)

for row in range(row_min, row_max + 1):

    for col in range(col_min, col_max + 1):

        url = f"{BASE_URL}/tile/{LEVEL}/{row}/{col}"

        response = requests.get(url, timeout=60)
        response.raise_for_status()

        result, raster, depth, nodata = decode_ma(
            response.content
        )

        if result != 0:
            raise ValueError(
                f"Could not decode tile {row}, {col}: {result}"
            )

        tile = np.ma.filled(
            raster,
            np.nan
        ).astype(np.float32)

        if tile.shape != (256, 256):
            raise ValueError(
                f"Unexpected tile dimensions: {tile.shape}"
            )

        r = (row - row_min) * TILE_SIZE
        c = (col - col_min) * TILE_SIZE

        mosaic[
            r:r + TILE_SIZE,
            c:c + TILE_SIZE
        ] = tile

        print(f"Downloaded tile: {row}, {col}")


left = ORIGIN_X + col_min * TILE_SPAN
top = ORIGIN_Y - row_min * TILE_SPAN

transform = from_origin(
    left,
    top,
    RESOLUTION,
    RESOLUTION
)


with rasterio.open(
    "noise_mosaic_temp.tif",
    "w",
    driver="GTiff",
    height=height,
    width=width,
    count=1,
    dtype="float32",
    crs="EPSG:3857",
    transform=transform,
    nodata=-9999
) as dst:

    dst.write(
        np.where(np.isfinite(mosaic), mosaic, -9999),
        1
    )


with rasterio.open("noise_mosaic_temp.tif") as src:

    geometry = box(xmin, ymin, xmax, ymax)

    clipped, clipped_transform = mask(
        src,
        [mapping(geometry)],
        crop=True,
        nodata=-9999
    )

    profile = src.profile.copy()

    profile.update(
        height=clipped.shape[1],
        width=clipped.shape[2],
        transform=clipped_transform,
        compress="deflate"
    )

    with rasterio.open(OUTPUT, "w", **profile) as dst:
        dst.write(clipped)

print("Finished:", OUTPUT)

Columns: 828 to 834
Rows: 764 to 769
Downloaded tile: 764, 828
Downloaded tile: 764, 829
Downloaded tile: 764, 830
Downloaded tile: 764, 831
Downloaded tile: 764, 832
Downloaded tile: 764, 833
Downloaded tile: 764, 834
Downloaded tile: 765, 828
Downloaded tile: 765, 829
Downloaded tile: 765, 830
Downloaded tile: 765, 831
Downloaded tile: 765, 832
Downloaded tile: 765, 833
Downloaded tile: 765, 834
Downloaded tile: 766, 828
Downloaded tile: 766, 829
Downloaded tile: 766, 830
Downloaded tile: 766, 831
Downloaded tile: 766, 832
Downloaded tile: 766, 833
Downloaded tile: 766, 834
Downloaded tile: 767, 828
Downloaded tile: 767, 829
Downloaded tile: 767, 830
Downloaded tile: 767, 831
Downloaded tile: 767, 832
Downloaded tile: 767, 833
Downloaded tile: 767, 834
Downloaded tile: 768, 828
Downloaded tile: 768, 829
Downloaded tile: 768, 830
Downloaded tile: 768, 831
Downloaded tile: 768, 832
Downloaded tile: 768, 833
Downloaded tile: 768, 834
Downloaded tile: 769, 828
Downloaded tile: 769, 829
D

In [12]:
import zipfile
import os

zip_path = (
    "/Users/sophiameyers/Documents/GitHub/"
    "30-day-mapping/data/"
    "CONUS_rail_road_and_aviation_noise_2022.zip"
)

print("ZIP size (GB):", round(os.path.getsize(zip_path) / 1e9, 2))

with zipfile.ZipFile(zip_path, "r") as z:
    files = z.namelist()

    print("\nTotal files:", len(files))

    for f in files[:50]:
        print(f)

ZIP size (GB): 1.63

Total files: 1245
CONUS_rail_road_and_aviation_noise_2022/
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000001.gdbindexes
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000001.gdbtable
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000001.gdbtablx
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000001.TablesByName.atx
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000002.gdbtable
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000002.gdbtablx
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000003.gdbindexes
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000003.gdbtable
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a0000

In [13]:
import zipfile
import os
from collections import Counter

zip_path = (
    "/Users/sophiameyers/Documents/GitHub/"
    "30-day-mapping/data/"
    "CONUS_rail_road_and_aviation_noise_2022.zip"
)

with zipfile.ZipFile(zip_path, "r") as z:

    files = z.infolist()

    folders = set()

    for f in files:
        parts = f.filename.split("/")
        for i in range(1, len(parts)):
            folders.add("/".join(parts[:i]))

    print("GEODATABASE FOLDERS:")
    for folder in sorted(folders):
        print(folder)

    print("\nLARGEST 30 FILES:")

    largest = sorted(
        [f for f in files if not f.is_dir()],
        key=lambda f: f.file_size,
        reverse=True
    )[:30]

    for f in largest:
        print(
            f"{f.file_size / 1e6:.2f} MB | {f.filename}"
        )

GEODATABASE FOLDERS:
CONUS_rail_road_and_aviation_noise_2022
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.Overviews
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.Overviews/CONUS_CONUS_rail_road_aviation_noise_mosaic.Overviews
CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb

LARGEST 30 FILES:
2419.17 MB | CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a000000d6.gdbtable
1225.85 MB | CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a0000001d.gdbtable
945.41 MB | CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000072.gdbtable
914.50 MB | CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a00000077.gdbtable
820.44 MB | CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a0000004f.gdbtable
808.80 MB | CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb/a0000006d.

In [16]:
import rasterio

zip_path = (
    "/Users/sophiameyers/Documents/GitHub/30-day-mapping/"
    "data/CONUS_rail_road_and_aviation_noise_2022.zip"
)

gdb_inside_zip = (
    "CONUS_rail_road_and_aviation_noise_2022/"
    "CONUS_rail_road_aviation_noise.gdb"
)

vsi_path = f"/vsizip/{zip_path}/{gdb_inside_zip}"

try:
    with rasterio.open(vsi_path) as src:
        print("Successfully opened!")
        print("Driver:", src.driver)
        print("Subdatasets:", src.subdatasets[:20])
        print("CRS:", src.crs)
        print("Bounds:", src.bounds)
except Exception as e:
    print(type(e).__name__, e)

Successfully opened!
Driver: OpenFileGDB
Subdatasets: ['OpenFileGDB:/vsizip//Users/sophiameyers/Documents/GitHub/30-day-mapping/data/CONUS_rail_road_and_aviation_noise_2022.zip/CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb:AL_rail_road_aviation_noise', 'OpenFileGDB:/vsizip//Users/sophiameyers/Documents/GitHub/30-day-mapping/data/CONUS_rail_road_and_aviation_noise_2022.zip/CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb:IA_rail_road_aviation_noise', 'OpenFileGDB:/vsizip//Users/sophiameyers/Documents/GitHub/30-day-mapping/data/CONUS_rail_road_and_aviation_noise_2022.zip/CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb:ID_rail_road_aviation_noise', 'OpenFileGDB:/vsizip//Users/sophiameyers/Documents/GitHub/30-day-mapping/data/CONUS_rail_road_and_aviation_noise_2022.zip/CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb:IL_rail_road_aviation_noise', 'OpenFileGDB:/vsizip//Users/sophiamey

/opt/anaconda3/envs/geospatial/lib/python3.14/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


In [17]:
import rasterio

with rasterio.open(vsi_path) as src:
    oregon_options = [
        s for s in src.subdatasets
        if "OR_rail_road_aviation_noise" in s
    ]

print("Oregon datasets found:", len(oregon_options))
for s in oregon_options:
    print(s)

Oregon datasets found: 1
OpenFileGDB:/vsizip//Users/sophiameyers/Documents/GitHub/30-day-mapping/data/CONUS_rail_road_and_aviation_noise_2022.zip/CONUS_rail_road_and_aviation_noise_2022/CONUS_rail_road_aviation_noise.gdb:OR_rail_road_aviation_noise


In [19]:
import os
import math
import rasterio
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds, Window

west, south, east, north = -122.85, 45.43, -122.45, 45.68

output_path = (
    "/Users/sophiameyers/Documents/GitHub/30-day-mapping/"
    "data/portland_transportation_noise_2022.tif"
)

os.makedirs(os.path.dirname(output_path), exist_ok=True)

with rasterio.open(oregon_path) as src:

    projected_bounds = transform_bounds(
        "EPSG:4326",
        src.crs,
        west, south, east, north,
        densify_pts=21
    )

    print("Portland bounds in raster CRS:", projected_bounds)

    raw_window = from_bounds(
        *projected_bounds,
        transform=src.transform
    )

    col_start = math.floor(raw_window.col_off)
    row_start = math.floor(raw_window.row_off)
    col_end = math.ceil(raw_window.col_off + raw_window.width)
    row_end = math.ceil(raw_window.row_off + raw_window.height)

    window = Window(
        col_start,
        row_start,
        col_end - col_start,
        row_end - row_start
    )

    window = window.intersection(
        Window(0, 0, src.width, src.height)
    )

    print("Crop dimensions:", window.width, "x", window.height)

    data = src.read(1, window=window)

    profile = src.profile.copy()
    profile.update(
        driver="GTiff",
        width=data.shape[1],
        height=data.shape[0],
        transform=src.window_transform(window),
        compress="DEFLATE",
        predictor=3,
        tiled=True,
        blockxsize=256,
        blockysize=256,
        BIGTIFF="IF_SAFER"
    )

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(data, 1)

print("Saved:", output_path)

Portland bounds in raster CRS: (-2072904.9953929193, 2779679.8397984887, -2035198.776821415, 2815021.5343642095)
Crop dimensions: 1258 x 1179
Saved: /Users/sophiameyers/Documents/GitHub/30-day-mapping/data/portland_transportation_noise_2022.tif


/var/folders/3t/qsrrrg8559d3byzjqfp_ybs40000gn/T/ipykernel_77470/291642289.py:57: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  data = src.read(1, window=window)


In [20]:
import rasterio
import numpy as np

with rasterio.open(output_path) as src:
    data = src.read(1, masked=True)

    print("CRS:", src.crs)
    print("Dimensions:", src.width, "x", src.height)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)
    print("Minimum noise value:", data.min())
    print("Maximum noise value:", data.max())
    print("File size (MB):", round(os.path.getsize(output_path) / 1e6, 2))

CRS: ESRI:102039
Dimensions: 1258 x 1179
Bounds: BoundingBox(left=-2072926.4675485217, bottom=2779675.1622865526, right=-2035186.4675485217, top=2815045.1622865526)
Resolution: (30.0, 30.0)
Minimum noise value: 45.0
Maximum noise value: 121.40001
File size (MB): 1.44


In [21]:
with rasterio.open(vsi_path) as src:
    for s in src.subdatasets:
        if s.startswith("OpenFileGDB:"):
            print(s.split(":")[-1])

AL_rail_road_aviation_noise
IA_rail_road_aviation_noise
ID_rail_road_aviation_noise
IL_rail_road_aviation_noise
IN_rail_road_aviation_noise
KS_rail_road_aviation_noise
KY_rail_road_aviation_noise
LA_rail_road_aviation_noise
MA_rail_road_aviation_noise
MD_rail_road_aviation_noise
ME_rail_road_aviation_noise
AR_rail_road_aviation_noise
MI_rail_road_aviation_noise
MN_rail_road_aviation_noise
MO_rail_road_aviation_noise
MS_rail_road_aviation_noise
MT_rail_road_aviation_noise
NC_rail_road_aviation_noise
ND_rail_road_aviation_noise
NE_rail_road_aviation_noise
NH_rail_road_aviation_noise
NJ_rail_road_aviation_noise
AZ_rail_road_aviation_noise
NM_rail_road_aviation_noise
NV_rail_road_aviation_noise
NY_rail_road_aviation_noise
OH_rail_road_aviation_noise
OK_rail_road_aviation_noise
OR_rail_road_aviation_noise
PA_rail_road_aviation_noise
RI_rail_road_aviation_noise
SC_rail_road_aviation_noise
SD_rail_road_aviation_noise
CA_rail_road_aviation_noise
TN_rail_road_aviation_noise
TX_rail_road_aviatio